# Per-residue SS and burial persistence over the MD trajectories

Fraction of frames (first 100 dropped) in which DSSP agrees with the static `ss_class` (`ss_persist`) and the amide N+H SASA agrees with the static `exposure` (`burial_persist`). `ss_ok` and `burial_ok` mark ≥ 0.9. No residue is dropped.

In [ ]:
import json, glob, os, re
import numpy as np
import mdtraj as mdt

RES = "../../data/md_1us"  # 1-us MD trajectories, one frame per ns
PRR = ("../../data/analysis/Fig4/sasa_results/per_residue_records.json")
OUT_DIR = "../../data/analysis/Fig4/per_protein_analysis"   # read by Fig 5c, 5d, 5f, S9, S10, S11

ENTRIES = ["11139_1", "36329_1", "30706_1", "6032_1", "16603_1", "10267_1", "6726_1", "52187_1", "19743_1", "11202_1",
           "34106_1", "4463_1", "4819_1", "5921_1", "6933_1", "16628_1", "34043_1", "26517_1", "34425_1", "11188_1",
           "19298_1", "25895_1", "52088_1", "16387_1", "5902_1", "6782_1", "10235_1", "19737_1", "17136_1", "34607_1",
           "50270_1", "6114_1", "25968_1", "5842_1", "27687_1", "16016_1", "25448_1", "36327_1", "51126_1", "17723_1",
           "16848_1", "4671_1", "5490_1", "34678_1", "19231_1", "18145_1", "16652_1", "16186_1"]

DROP = 100                    # equilibration frames dropped (1 ns/frame)
PERSIST_CUT = 0.90            # threshold for both SS and burial persistence
DSSP_OF = {"sheet": ("E", "B"), "helix": ("H", "G", "I")}   # coil = complement (T, S, blank)
BACKBONE_H_NAMES = ("H", "HN", "H1")   # backbone amide H atom name across force fields

_FRAME_RE = re.compile(r"^frame_?(\d+)")

def _frames(e):
    return sorted((p for p in glob.glob(f"{RES}/{e}/frame*.pdb")
                   if _FRAME_RE.match(os.path.basename(p))),
                  key=lambda p: int(_FRAME_RE.match(os.path.basename(p)).group(1)))

def load_traj(paths):
    """Load a list of single-frame PDBs into one Trajectory, ignoring unit-cell/box info.

    Some dumped frames (e.g. a stray frame_0002.pdb) are missing their CRYST1 record while their
    neighbors have one; mdt.load(list_of_paths) joins frames pairwise and raises "Mixing
    trajectories with and without unitcell" the moment it hits one of those. We don't need PBC
    box info for DSSP or per-atom SASA on a single monomer, so build the trajectory from raw
    coordinates only (first frame's topology, assumed identical atom composition across frames)
    and skip the box check entirely.
    """
    frames = [mdt.load(p) for p in paths]
    xyz = np.concatenate([f.xyz for f in frames], axis=0)
    return mdt.Trajectory(xyz=xyz, topology=frames[0].topology)

REC = json.load(open(PRR))

## Main pass — one trajectory at a time

In [ ]:
rows = []

for e in ENTRIES:
    rec = REC[e]
    L = len(rec)

    win = load_traj(_frames(e))[DROP:]
    top = win.top
    n = min(top.n_residues, L)

    # ---- SS persistence: DSSP over the trajectory ----
    d = mdt.compute_dssp(win, simplified=False)                     # [n_frames, n_res]
    frac_sheet = np.isin(d[:, :n], DSSP_OF["sheet"]).mean(0)
    frac_helix = np.isin(d[:, :n], DSSP_OF["helix"]).mean(0)
    frac_coil  = 1.0 - frac_sheet - frac_helix

    # ---- burial persistence: per-frame backbone amide N+H SASA ----
    sasa = mdt.shrake_rupley(win, mode="atom")                      # [n_frames, n_atoms], nm^2
    exposed_frac = np.full(n, np.nan)
    for r in range(n):
        res = top.residue(r)
        n_idx = next((a.index for a in res.atoms if a.name == "N"), None)
        h_idx = next((a.index for a in res.atoms if a.name in BACKBONE_H_NAMES), None)
        if n_idx is None or h_idx is None:
            continue    # proline / no amide H -> burial persistence undefined for this residue
        nh_sasa = sasa[:, n_idx] + sasa[:, h_idx]
        exposed_frac[r] = (nh_sasa > 0).mean()

    # ---- compare against the static labels in per_residue_records.json ----
    for i in range(n):
        static_ss  = rec[i].get("ss_class")
        static_exp = rec[i].get("exposure")

        ss_persist = {"sheet": frac_sheet[i], "helix": frac_helix[i], "coil": frac_coil[i]}.get(static_ss)
        ss_ok = ss_persist is not None and ss_persist >= PERSIST_CUT

        if static_exp == "buried":
            burial_persist = 1.0 - exposed_frac[i]
        elif static_exp == "exposed":
            burial_persist = exposed_frac[i]
        else:
            burial_persist = np.nan
        burial_ok = np.isfinite(burial_persist) and burial_persist >= PERSIST_CUT

        rows.append(dict(
            entry=e, resnum=rec[i]["resnum"], residue_type=rec[i]["residue_type"],
            ss_class=static_ss,
            ss_persist=None if ss_persist is None else round(float(ss_persist), 4),
            ss_ok=bool(ss_ok),
            exposure=static_exp,
            burial_persist=None if not np.isfinite(burial_persist) else round(float(burial_persist), 4),
            burial_ok=bool(burial_ok),
        ))

    print(f"{e}: n_frames={win.n_frames}  n_res={n}", flush=True)

## Save

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)

print(f"{len(df)} residues from {df.entry.nunique()} proteins")
print("\nss_ok counts by ss_class:")
print(df.groupby("ss_class")["ss_ok"].value_counts().unstack(fill_value=0))
print("\nburial_ok counts by exposure:")
print(df.groupby("exposure")["burial_ok"].value_counts().unstack(fill_value=0))
print("\nboth ss_ok and burial_ok:", int((df.ss_ok & df.burial_ok).sum()))

CSV_OUT = f"{OUT_DIR}/persist90_ss_burial_simple.csv"
df.to_csv(CSV_OUT, index=False)
print(f"\nwrote {CSV_OUT}  ({len(df)} rows)")